## Notebook 10 — Media Behavior Baseline and Cluster Deviations

This notebook analyzes **media behavior patterns across the synthetic U.S. population** produced in the previous stages of the segmentation pipeline.

The objective is to identify **media behaviors that characterize each structural population cluster** by comparing cluster-level media usage patterns against the **national media baseline**.

The analysis proceeds in three main steps:

1. **Load the media-enriched population dataset**  
   The individual-level dataset generated in Notebook 09 is loaded. This dataset contains the ACS-based structural population enriched with projected media behavior probabilities.

2. **Compute the national media baseline**  
   Using ACS population weights (`pwgtp`), national baseline probabilities are calculated for each media behavior variable.

3. **Estimate cluster-level deviations from the national baseline**  
   Cluster-level weighted averages are computed and compared to the national baseline in order to identify media behaviors that **meaningfully distinguish each cluster from the overall population**.

The resulting deviation profiles highlight the **information environments associated with each structural segment**.

These media behavior signatures can then be interpreted alongside the **psychological cluster signatures derived in Notebook 08**, providing a combined view of:

- structural characteristics  
- psychological traits  
- media behavior patterns  

Together, these dimensions support the interpretation of **behaviorally grounded audience segments and proto-personas**.

In [1]:
#import
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
# ------------------------------------------------------------
# Project root and data directories
# ------------------------------------------------------------

from pathlib import Path

# project root (notebooks are inside /notebooks)
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/marcomagnolo/Projects/us-audience-segmentation


In [3]:
# ------------------------------------------------------------
# Project root and data directories
# ------------------------------------------------------------

from pathlib import Path

# project root (notebooks are inside /notebooks)
PROJECT_ROOT = Path.cwd().parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)

PROJECT_ROOT: /Users/marcomagnolo/Projects/us-audience-segmentation
DATA_PROCESSED: /Users/marcomagnolo/Projects/us-audience-segmentation/data/processed


In [4]:
# ------------------------------------------------------------
# Load media-enriched population dataset
# ------------------------------------------------------------

MEDIA_POP_PATH = DATA_PROCESSED / "09_us_structural_media_population_v1.parquet"

if not MEDIA_POP_PATH.exists():
    raise FileNotFoundError(
        "Media-enriched population dataset not found. Run Notebook 09 first."
    )

df = pd.read_parquet(MEDIA_POP_PATH)

print("Dataset loaded")
print("Shape:", df.shape)
print("Source:", MEDIA_POP_PATH)

Dataset loaded
Shape: (778466, 50)
Source: /Users/marcomagnolo/Projects/us-audience-segmentation/data/processed/09_us_structural_media_population_v1.parquet


In [5]:
media_cols = [
    "internet_access",
    "mobile_internet_access",
    "internet_frequency",
    "radio_listener",
    "youtube",
    "facebook",
    "instagram",
    "tiktok",
    "whatsapp",
    "reddit",
    "snapchat",
    "x_twitter",
    "threads",
    "bluesky",
    "truthsocial"
]

In [6]:
# computing national media baseline
national_media_baseline = (
    df[media_cols]
    .multiply(df["pwgtp"], axis=0)
    .sum()
    / df["pwgtp"].sum()
)

national_media_baseline

internet_access           0.946541
mobile_internet_access    0.939734
internet_frequency        1.772246
radio_listener            0.681662
youtube                   0.851719
facebook                  0.729095
instagram                 0.505528
tiktok                    0.401325
whatsapp                  0.331104
reddit                    0.267975
snapchat                  0.291080
x_twitter                 0.206566
threads                   0.091674
bluesky                   0.039250
truthsocial               0.034554
dtype: float64

In [7]:
# reshaping for visualization
baseline_df = (
    national_media_baseline
    .reset_index()
)

baseline_df.columns = ["media_trait", "national_baseline"]

baseline_df

,media_trait,national_baseline
0,internet_access,0.946541
1,mobile_internet_access,0.939734
2,internet_frequency,1.772246
3,radio_listener,0.681662
4,youtube,0.851719
5,facebook,0.729095
6,instagram,0.505528
7,tiktok,0.401325
8,whatsapp,0.331104
9,reddit,0.267975


In [8]:
# ------------------------------------------------------------
# Save national media baseline
# ------------------------------------------------------------

BASELINE_PATH = DATA_PROCESSED / "10_us_media_national_baseline_v1.csv"

baseline_df.to_csv(BASELINE_PATH, index=False)

print("Saved:", BASELINE_PATH)
print("Shape:", baseline_df.shape)

Saved: /Users/marcomagnolo/Projects/us-audience-segmentation/data/processed/10_us_media_national_baseline_v1.csv
Shape: (15, 2)


In [9]:
# computing cluster media means weighted by population
cluster_media_means = (
    df
    .groupby("cluster")
    .apply(
        lambda g: (
            g[media_cols]
            .multiply(g["pwgtp"], axis=0)
            .sum()
            / g["pwgtp"].sum()
        )
    )
)

cluster_media_means

,internet_access,mobile_internet_access,internet_frequency,radio_listener,youtube,facebook,instagram,tiktok,whatsapp,reddit,snapchat,x_twitter,threads,bluesky,truthsocial
cluster,,,,,,,,,,,,,,,
0,0.975454,0.974340,1.618052,0.691773,0.907438,0.773655,0.564019,0.422456,0.399770,0.307896,0.308839,0.229114,0.098892,0.041094,0.034684
1,0.972807,0.963741,1.683968,0.712918,0.882664,0.751972,0.523022,0.379944,0.318972,0.297572,0.286454,0.220336,0.088710,0.046609,0.043942
2,0.946532,0.956776,1.743077,0.633204,0.881773,0.734379,0.569553,0.507429,0.429437,0.242459,0.338328,0.211061,0.107695,0.032119,0.032088
3,0.942647,0.944523,1.695227,0.629829,0.876257,0.727012,0.581149,0.480105,0.332724,0.290661,0.373130,0.214134,0.092960,0.042641,0.029964
4,0.886165,0.855681,2.201932,0.726316,0.701914,0.622916,0.268370,0.216406,0.221912,0.141956,0.121506,0.134269,0.049340,0.026155,0.035446
5,0.943615,0.949989,1.687194,0.609690,0.897202,0.734718,0.598896,0.559563,0.390538,0.281758,0.399730,0.222241,0.123960,0.035508,0.025173
6,0.944733,0.942673,1.728542,0.662074,0.858327,0.748480,0.544986,0.442786,0.337588,0.289900,0.329574,0.219701,0.107563,0.040647,0.029392


In [10]:
# compute deviations from national baseline
cluster_media_deviation = cluster_media_means - national_media_baseline

In [11]:
cluster_media_dev_long = (
    cluster_media_deviation
    .reset_index()
    .melt(
        id_vars="cluster",
        var_name="trait",
        value_name="deviation"
    )
)

cluster_media_dev_long.head()

,cluster,trait,deviation
0,0,internet_access,0.028913
1,1,internet_access,0.026266
2,2,internet_access,-0.000008
3,3,internet_access,-0.003894
4,4,internet_access,-0.060375


In [12]:
cluster_media_dev_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   cluster    105 non-null    uint16 
 1   trait      105 non-null    str    
 2   deviation  105 non-null    float64
dtypes: float64(1), str(1), uint16(1)
memory usage: 3.0 KB


In [13]:
cluster_media_dev_long["deviation"].describe()

count    105.000000
mean       0.003873
std        0.074314
min       -0.237158
25%       -0.007132
50%        0.007218
75%        0.030054
max        0.429686
Name: deviation, dtype: float64

In [14]:
# based on the distribution of deviations, we can set a threshold for "significant" deviation (3%)
threshold = 0.03

cluster_media_signature = (
    cluster_media_dev_long
    .loc[cluster_media_dev_long["deviation"].abs() >= threshold]
    .copy()
)

In [15]:
# sorting
cluster_media_signature["abs_dev"] = cluster_media_signature["deviation"].abs()

cluster_media_signature.sort_values(
    ["cluster","abs_dev"],
    ascending=[True,False],
    inplace=True
)
cluster_media_signature

,cluster,trait,deviation,abs_dev
14,0,internet_frequency,-0.154194,0.154194
56,0,whatsapp,0.068666,0.068666
42,0,instagram,0.058491,0.058491
28,0,youtube,0.055719,0.055719
35,0,facebook,0.044560,0.044560
63,0,reddit,0.039921,0.039921
7,0,mobile_internet_access,0.034607,0.034607
15,1,internet_frequency,-0.088278,0.088278
22,1,radio_listener,0.031256,0.031256
29,1,youtube,0.030945,0.030945


### Media Behavior Deviations by Cluster

In this step, media behavior patterns were analyzed to identify platform usage characteristics that distinguish each structural cluster from the national population baseline.

First, a **national media baseline** was computed using the PWGTP population weights from the ACS-enriched synthetic population dataset. This baseline represents the estimated share of the U.S. population using each media platform or exhibiting each media behavior.

Second, **cluster-level weighted means** were calculated for all media variables. Deviations were then computed as:

cluster_mean − national_baseline

This produced a measure of **behavioral over- or under-indexing** for each cluster relative to the national population.

To isolate meaningful signals and remove noise, a **±0.03 deviation threshold (3 percentage points)** was applied, consistent with the filtering logic used previously for psychological traits. Only media behaviors exceeding this threshold were retained as **cluster-defining media signals**.

### Outcome

The resulting table captures the **media behavioral signature of each cluster**, highlighting platforms and behaviors where clusters significantly diverge from the national baseline. These signals reveal distinct patterns of digital engagement across clusters, such as stronger usage of specific social media platforms or lower overall internet activity.

These media signatures will be combined with the previously derived **psychological cluster signatures** to construct the final **audience archetypes** in the next stage of the analysis.

In [16]:
# for better readability, we can combine the trait name with the media usage prefix
cluster_media_signature["trait"] = "media_usage: " + cluster_media_signature["trait"]

In [17]:
# adding direction
cluster_media_signature["direction"] = np.where(
    cluster_media_signature["deviation"] > 0,
    "above baseline",
    "below baseline"
)

In [18]:
# align with psych traits
cluster_media_signature = cluster_media_signature[
    ["cluster","trait","deviation","direction"]
]

In [19]:
cluster_media_signature.head()

,cluster,trait,deviation,direction
14,0,media_usage: internet_frequency,-0.154194,below baseline
56,0,media_usage: whatsapp,0.068666,above baseline
42,0,media_usage: instagram,0.058491,above baseline
28,0,media_usage: youtube,0.055719,above baseline
35,0,media_usage: facebook,0.044560,above baseline


In [20]:
# ------------------------------------------------------------
# Load cluster psychological signatures
# ------------------------------------------------------------

PSYCH_PATH = DATA_PROCESSED / "08_us_cluster_psychological_signatures_v1.parquet"

if not PSYCH_PATH.exists():
    raise FileNotFoundError(
        "Cluster psychological signatures not found. Run Notebook 08 first."
    )

cluster_psych_signature = pd.read_parquet(PSYCH_PATH)

print("Loaded:", PSYCH_PATH)
print("Shape:", cluster_psych_signature.shape)

cluster_psych_signature.head()

Loaded: /Users/marcomagnolo/Projects/us-audience-segmentation/data/processed/08_us_cluster_psychological_signatures_v1.parquet
Shape: (30, 4)


,cluster,trait,deviation,direction
0,1,party_alignment: republican,0.040,above baseline
1,2,media_engagement: low,0.085,above baseline
2,2,media_engagement: high,-0.076,below baseline
3,2,party_alignment: republican,-0.065,below baseline
4,2,party_alignment: independent,0.051,above baseline


In [21]:
cluster_psych_signature = cluster_psych_signature.copy()
cluster_media_signature = cluster_media_signature.copy()

cluster_psych_signature["source"] = "psych"
cluster_media_signature["source"] = "media"

In [22]:
# merging media and psych signatures
cluster_full_signature = pd.concat(
    [cluster_psych_signature, cluster_media_signature],
    ignore_index=True
)

In [23]:
# sort by signal strength
cluster_full_signature["abs_dev"] = cluster_full_signature["deviation"].abs()

cluster_full_signature.sort_values(
    ["cluster", "abs_dev"],
    ascending=[True, False],
    inplace=True
)

In [24]:
cluster_full_signature.head(20)

,cluster,trait,deviation,direction,source,abs_dev
30,0,media_usage: internet_frequency,-0.154194,below baseline,media,0.154194
31,0,media_usage: whatsapp,0.068666,above baseline,media,0.068666
32,0,media_usage: instagram,0.058491,above baseline,media,0.058491
33,0,media_usage: youtube,0.055719,above baseline,media,0.055719
34,0,media_usage: facebook,0.044560,above baseline,media,0.044560
35,0,media_usage: reddit,0.039921,above baseline,media,0.039921
36,0,media_usage: mobile_internet_access,0.034607,above baseline,media,0.034607
37,1,media_usage: internet_frequency,-0.088278,below baseline,media,0.088278
0,1,party_alignment: republican,0.040000,above baseline,psych,0.040000
38,1,media_usage: radio_listener,0.031256,above baseline,media,0.031256


In [25]:
# ANALYSIS

In [26]:
# selecting top traits per cluster and source
top_psych = (
    cluster_full_signature
    .query("source == 'psych'")
    .groupby("cluster")
    .head(3)
)

top_media = (
    cluster_full_signature
    .query("source == 'media'")
    .groupby("cluster")
    .head(3)
)

In [27]:
top_signals = pd.concat(
    [top_psych, top_media],
    ignore_index=True
)

In [28]:
top_signals.head(20)

,cluster,trait,deviation,direction,source,abs_dev
0,1,party_alignment: republican,0.040000,above baseline,psych,0.040000
1,2,media_engagement: low,0.085000,above baseline,psych,0.085000
2,2,media_engagement: high,-0.076000,below baseline,psych,0.076000
3,2,party_alignment: republican,-0.065000,below baseline,psych,0.065000
4,3,media_engagement: high,-0.070000,below baseline,psych,0.070000
5,3,media_engagement: low,0.070000,above baseline,psych,0.070000
6,3,life_satisfaction: pretty_happy,0.038000,above baseline,psych,0.038000
7,4,media_engagement: low,-0.116000,below baseline,psych,0.116000
8,4,media_engagement: high,0.099000,above baseline,psych,0.099000
9,4,ideology: conservative,0.074000,above baseline,psych,0.074000


In [29]:
# sort for readability
top_signals = top_signals.sort_values(
    ["cluster", "abs_dev"],
    ascending=[True, False]
)

In [30]:
# clean summary table
cluster_summary = (
    top_signals
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="key_traits")
)

cluster_summary

,cluster,key_traits
0,0,"[media_usage: internet_frequency, media_usage:..."
1,1,"[media_usage: internet_frequency, party_alignm..."
2,2,"[media_usage: tiktok, media_usage: whatsapp, m..."
3,3,"[media_usage: snapchat, media_usage: tiktok, m..."
4,4,"[media_usage: internet_frequency, media_usage:..."
5,5,"[media_usage: tiktok, media_usage: snapchat, m..."
6,6,"[media_usage: internet_frequency, media_usage:..."


### FINAL SEGMENTATION TABLE OF THE WHOLE PIPELIN

In [31]:
# ------------------------------------------------------------
# Load clustered structural population dataset
# ------------------------------------------------------------

CLUSTER_POP_PATH = DATA_PROCESSED / "us_structural_population_clustered_v1.parquet"

if not CLUSTER_POP_PATH.exists():
    raise FileNotFoundError(
        "Clustered structural population dataset not found. Run Notebook 06 first."
    )

df_struct = pd.read_parquet(CLUSTER_POP_PATH)

print("Loaded:", CLUSTER_POP_PATH)
print("Shape:", df_struct.shape)

df_struct.head()

Loaded: /Users/marcomagnolo/Projects/us-audience-segmentation/data/processed/us_structural_population_clustered_v1.parquet
Shape: (1000000, 18)


,serialno,sporder,pwgtp,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,mar_tier,commute_tier,tenure,household_size,vehicle_count,puma,hhincome_tier,household_type,cluster
0,2023HU1043211,2,58,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,2,0,4316,0-19k,housing_unit,6
1,2019HU1076190,2,46,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,4,2,5922,20-49k,housing_unit,0
2,2019GQ0046130,1,12,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,Never_Married,Missing,Group_Quarters,1,0,11300,group_quarters,group_quarters,6
3,2019HU0403832,1,76,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Work_From_Home,Owner,5,2,2510,50-99k,housing_unit,0
4,2019HU0277198,1,64,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,4,1,4607,20-49k,housing_unit,5


In [32]:
# computing weighted cluster population
cluster_population = (
    df_struct
    .groupby("cluster")["pwgtp"]
    .sum()
    .reset_index(name="weighted_population")
)

cluster_population

,cluster,weighted_population
0,0,5255739
1,1,7253101
2,2,2892345
3,3,4486033
4,4,5253120
5,5,8465027
6,6,7339940


In [33]:
# compute population share
cluster_population["population_share"] = (
    cluster_population["weighted_population"]
    / cluster_population["weighted_population"].sum()
)

In [34]:
# keeping final fields
cluster_population = cluster_population[
    ["cluster", "population_share"]
]

cluster_population

,cluster,population_share
0,0,0.128360
1,1,0.177141
2,2,0.070639
3,3,0.109562
4,4,0.128296
5,5,0.206740
6,6,0.179262


In [35]:
# check
cluster_population["population_share"].sum()

np.float64(1.0)

In [36]:
# create psych table
psych_traits = (
    top_signals
    .query("source == 'psych'")
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="psych_signals")
)

psych_traits

,cluster,psych_signals
0,1,[party_alignment: republican]
1,2,"[media_engagement: low, media_engagement: high..."
2,3,"[media_engagement: high, media_engagement: low..."
3,4,"[media_engagement: low, media_engagement: high..."
4,5,"[media_engagement: low, media_engagement: high..."
5,6,"[party_alignment: republican, party_alignment:..."


In [37]:
# create media table
media_traits = (
    top_signals
    .query("source == 'media'")
    .groupby("cluster")["trait"]
    .apply(list)
    .reset_index(name="media_signals")
)

media_traits

,cluster,media_signals
0,0,"[media_usage: internet_frequency, media_usage:..."
1,1,"[media_usage: internet_frequency, media_usage:..."
2,2,"[media_usage: tiktok, media_usage: whatsapp, m..."
3,3,"[media_usage: snapchat, media_usage: tiktok, m..."
4,4,"[media_usage: internet_frequency, media_usage:..."
5,5,"[media_usage: tiktok, media_usage: snapchat, m..."
6,6,"[media_usage: internet_frequency, media_usage:..."


In [38]:
# building final cluster summary table
cluster_profiles = (
    cluster_population
    .merge(psych_traits, on="cluster", how="left")
    .merge(media_traits, on="cluster", how="left")
)

cluster_profiles

,cluster,population_share,psych_signals,media_signals
0,0,0.128360,NaN,"[media_usage: internet_frequency, media_usage:..."
1,1,0.177141,[party_alignment: republican],"[media_usage: internet_frequency, media_usage:..."
2,2,0.070639,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: whatsapp, m..."
3,3,0.109562,"[media_engagement: high, media_engagement: low...","[media_usage: snapchat, media_usage: tiktok, m..."
4,4,0.128296,"[media_engagement: low, media_engagement: high...","[media_usage: internet_frequency, media_usage:..."
5,5,0.206740,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: snapchat, m..."
6,6,0.179262,"[party_alignment: republican, party_alignment:...","[media_usage: internet_frequency, media_usage:..."


In [39]:
# ------------------------------------------------------------
# Save final cluster profile table
# ------------------------------------------------------------

OUTPUT_PATH = DATA_PROCESSED / "10_us_cluster_profiles_v1.parquet"

cluster_profiles.to_parquet(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", cluster_profiles.shape)

Saved: /Users/marcomagnolo/Projects/us-audience-segmentation/data/processed/10_us_cluster_profiles_v1.parquet
Shape: (7, 4)
